# 02 有限差分公式

本节系统整理一阶数值微分中最常用的有限差分公式。重点不是只记住系数，而是看到系数如何由 Taylor 展开或局部插值多项式得到。


In [ ]:
import math
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["font.sans-serif"] = [
    "Arial Unicode MS",
    "PingFang SC",
    "Heiti SC",
    "SimHei",
    "Noto Sans CJK SC",
]
plt.rcParams["axes.unicode_minus"] = False

ROOT = pathlib.Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "pyproject.toml").exists() and (candidate / "src" / "py_sc").exists():
        ROOT = candidate
        break
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from py_sc import (
    backward_difference,
    central_difference,
    five_point_center_derivative,
    five_point_endpoint_derivative,
    forward_difference,
    observed_order,
    three_point_endpoint_derivative,
)


## 前向、后向和中心差分

由 Taylor 展开可得

$$
\frac{f(x+h)-f(x)}{h}=f'(x)+O(h),
$$

$$
\frac{f(x)-f(x-h)}{h}=f'(x)+O(h),
$$

$$
\frac{f(x+h)-f(x-h)}{2h}=f'(x)+O(h^2).
$$

因此单边差分适合边界或只能向一侧取值的情况，中心差分适合内部节点。


In [ ]:
def teaching_forward(f, x, h):
    return (f(x + h) - f(x)) / h


def teaching_backward(f, x, h):
    return (f(x) - f(x - h)) / h


def teaching_central(f, x, h):
    return (f(x + h) - f(x - h)) / (2 * h)

x0 = 0.7
exact = math.cos(x0)
h_values = 2.0 ** (-np.arange(3, 10, dtype=float))
rows = []
for h in h_values:
    rows.append([
        h,
        abs(teaching_forward(math.sin, x0, h) - exact),
        abs(teaching_backward(math.sin, x0, h) - exact),
        abs(teaching_central(math.sin, x0, h) - exact),
    ])
rows = np.array(rows)
print("h, forward_error, backward_error, central_error")
print(rows)
print("中心差分实验阶：", observed_order(rows[:, 3]))


## 由 Taylor 展开求差分系数

以模板 $x+s_jh$ 为例，希望

$$
f'(x)\approx \frac{1}{h}\sum_j c_j f(x+s_jh).
$$

把每个 $f(x+s_jh)$ 展开，要求常数项系数为 $0$，一阶项系数为 $1$，并尽量让更高阶项为 $0$，就得到一个线性方程组。


In [ ]:
def teaching_taylor_weights(offsets, derivative_order=1):
    offsets = np.asarray(offsets, dtype=float)
    n = offsets.size
    matrix = np.vander(offsets, N=n, increasing=True).T
    rhs = np.zeros(n)
    rhs[derivative_order] = math.factorial(derivative_order)
    return np.linalg.solve(matrix, rhs)

print("三点中心一阶系数：", teaching_taylor_weights([-1, 0, 1], 1))
print("三点左端点一阶系数：", teaching_taylor_weights([0, 1, 2], 1))
print("五点中心一阶系数：", teaching_taylor_weights([-2, -1, 0, 1, 2], 1))


## 三点端点公式

左端点公式为

$$
f'(x)\approx\frac{-3f(x)+4f(x+h)-f(x+2h)}{2h},
$$

右端点公式为

$$
f'(x)\approx\frac{3f(x)-4f(x-h)+f(x-2h)}{2h}.
$$

它们在边界处比简单前向或后向差分更精确，但需要更多函数值。


In [ ]:
def teaching_three_point_left(f, x, h):
    return (-3 * f(x) + 4 * f(x + h) - f(x + 2 * h)) / (2 * h)


def teaching_three_point_right(f, x, h):
    return (3 * f(x) - 4 * f(x - h) + f(x - 2 * h)) / (2 * h)

h = 0.1
print(teaching_three_point_left(math.sin, 0.0, h), "exact=", 1.0)
print(float(three_point_endpoint_derivative(np.sin, 0.0, h, side="left")))


## 五点中心和五点单边公式

五点中心公式为

$$
f'(x)\approx
\frac{f(x-2h)-8f(x-h)+8f(x+h)-f(x+2h)}{12h},
$$

对足够光滑的函数具有四阶截断误差。五点单边公式可用于端点，但函数计算次数更多，对函数值扰动也更敏感。


In [ ]:
def teaching_five_point_center(f, x, h):
    return (f(x - 2 * h) - 8 * f(x - h) + 8 * f(x + h) - f(x + 2 * h)) / (12 * h)

x0 = 0.7
exact = math.cos(x0)
errors_central = []
errors_five = []
for h in h_values:
    errors_central.append(abs(teaching_central(math.sin, x0, h) - exact))
    errors_five.append(abs(teaching_five_point_center(math.sin, x0, h) - exact))

print("中心差分阶：", observed_order(np.array(errors_central)))
print("五点中心阶：", observed_order(np.array(errors_five)))
print("正式五点实现：", float(five_point_center_derivative(np.sin, x0, 0.05)))
print("五点右端点实现：", float(five_point_endpoint_derivative(np.sin, math.pi, 0.05, side="right")))


## 误差图像与结果分析

在光滑函数上，前向和后向差分的误差大致随 $h$ 一阶下降，中心差分随 $h^2$ 下降，五点中心公式随 $h^4$ 下降。实际计算中，当 $h$ 继续变小到舍入误差主导区域，高阶公式也不会无限改进。


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(h_values, errors_central, "o-", label="中心差分")
ax.loglog(h_values, errors_five, "s-", label="五点中心")
ax.invert_xaxis()
ax.set_xlabel("步长 h")
ax.set_ylabel("绝对误差")
ax.set_title("中心差分与五点中心公式误差比较")
ax.grid(True, which="both", alpha=0.3)
ax.legend()
plt.show()


## 小结

有限差分公式的本质是局部多项式信息的线性组合。更多点通常能提高截断误差阶，但也会增加函数计算次数、边界处理复杂度和对数据扰动的敏感性。因此公式选择要同时看位置、光滑性、函数计算成本和误差来源。
